# GTEx model with different priors

## Load libraries

In [4]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

## Output directory

In [5]:
output_data_dir <- config$GTEx$DATASET_FOLDER
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# GTEx model with different priors

In [27]:
options(repr.matrix.max.rows = Inf)

# List all saved models, omitting "prior_PLIER" and "all"
rds_files <- list.files(output_data_dir, pattern = "^gtex_.*_CLAMP\\.rds$", full.names = TRUE)
rds_files <- rds_files[!grepl("prior_PLIER|all", rds_files)]

summaries <- list()

for (file in rds_files) {
  prior_name <- sub("^gtex_(.*)_CLAMP\\.rds$", "\\1", basename(file))
  message("Reading summary for: ", prior_name)

  res <- readRDS(file)
  if (!is.null(res$summary)) {
    summaries[[prior_name]] <- res$summary %>%
      dplyr::filter(FDR < 0.05 & AUC > 0.7)
  }
}

# Combine into one data frame with prior name column
all_summaries <- dplyr::bind_rows(
  lapply(names(summaries), function(p) {
    dplyr::mutate(summaries[[p]], prior = p)
  })
)

Reading summary for: CellMarker_2024



Reading summary for: Chromosome_Location

Reading summary for: GO_Biological_Process_2025

Reading summary for: GTEx_Tissues_V8_2023

Reading summary for: GWAS_Catalog_2025

Reading summary for: Human_Gene_Atlas

Reading summary for: KEGG_2021_Human

Reading summary for: LINCS_L1000_CRISPR_KO_Consensus_Sigs

Reading summary for: Metabolomics_Workbench_Metabolites_2022

Reading summary for: OMIM_Disease

Reading summary for: Proteomics_Drug_Atlas_2023

Reading summary for: TF_Perturbations_Followed_by_Expression

Reading summary for: UK_Biobank_GWAS_v1



In [28]:
all_summaries %>%
    dplyr::mutate(pathway_prior = paste0(prior, '_', pathway)) %>%
    dplyr::select(LV, pathway_prior) %>%
    dplyr::group_by(LV) %>%
    dplyr::summarise(pathway_prior = paste(pathway_prior, collapse = ', '), .groups = "drop") %>%
    dplyr::arrange(as.numeric(gsub("[^0-9]", "", LV)))

LV,pathway_prior
<chr>,<chr>
LV1,"Human_Gene_Atlas_UterusCorpus, TF_Perturbations_Followed_by_Expression_MAF OE MACROPHAGE HUMAN GSE98368 RNASEQ DOWN"
LV2,"GO_Biological_Process_2025_Synaptic Vesicle Cycle (GO:0099504), GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Male 50-59 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Female 60-69 Up, GTEx_Tissues_V8_2023_Brain - Anterior Cingulate Cortex (BA24) Female 50-59 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Male 20-29 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Male 70-79 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Female 40-49 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Female 50-59 Up, GTEx_Tissues_V8_2023_Brain - Frontal Cortex (BA9) Male 40-49 Up, GTEx_Tissues_V8_2023_Brain - Anterior Cingulate Cortex (BA24) Female 20-29 Up, Human_Gene_Atlas_Cerebellum, Human_Gene_Atlas_Thalamus, Human_Gene_Atlas_Wholebrain, Human_Gene_Atlas_Fetalbrain, Human_Gene_Atlas_OccipitalLobe, Human_Gene_Atlas_Amygdala, Human_Gene_Atlas_PrefrontalCortex, KEGG_2021_Human_GABAergic synapse, KEGG_2021_Human_Insulin secretion, KEGG_2021_Human_Nicotine addiction, KEGG_2021_Human_Synaptic vesicle cycle, OMIM_Disease_epilepsy, TF_Perturbations_Followed_by_Expression_HIVEP2 KO MOUSE GSE41307 CREEDSID GENE 3094 UP, TF_Perturbations_Followed_by_Expression_NFIA DEFICIENCY MOUSE GSE4623 CREEDSID GENE 667 DOWN, TF_Perturbations_Followed_by_Expression_MEF2A KD MOUSE GSE47150 CREEDSID GENE 957 DOWN, TF_Perturbations_Followed_by_Expression_MECP2 KD MOUSE GSE47150 CREEDSID GENE 951 DOWN, TF_Perturbations_Followed_by_Expression_MEF2D KD MOUSE GSE47150 CREEDSID GENE 956 DOWN"
LV3,"CellMarker_2024_Neutrophil Kidney Human, CellMarker_2024_Neutrophil Peripheral Blood Human, CellMarker_2024_Neutrophil Stomach Human, GTEx_Tissues_V8_2023_Whole Blood Female 50-59 Up, GTEx_Tissues_V8_2023_Whole Blood Male 50-59 Up, GTEx_Tissues_V8_2023_Whole Blood Female 20-29 Up, GTEx_Tissues_V8_2023_Whole Blood Male 30-39 Up, GTEx_Tissues_V8_2023_Whole Blood Female 30-39 Up, GTEx_Tissues_V8_2023_Whole Blood Male 20-29 Up, GTEx_Tissues_V8_2023_Whole Blood Female 40-49 Up, Human_Gene_Atlas_CD33+ Myeloid, Human_Gene_Atlas_Bonemarrow, Human_Gene_Atlas_WholeBlood, TF_Perturbations_Followed_by_Expression_EZH2 SHRNA PROE HUMAN GSE59089 RNASEQ UP"
LV4,"CellMarker_2024_Fibroblast Undefined Mouse, GTEx_Tissues_V8_2023_Adipose - Subcutaneous Male 50-59 Up, GTEx_Tissues_V8_2023_Adipose - Subcutaneous Male 60-69 Up, GTEx_Tissues_V8_2023_Adipose - Subcutaneous Male 40-49 Up, GTEx_Tissues_V8_2023_Cells - Cultured Fibroblasts Male 30-39 Up, GTEx_Tissues_V8_2023_Adipose - Subcutaneous Female 50-59 Up, GTEx_Tissues_V8_2023_Breast - Mammary Tissue Male 70-79 Up, GTEx_Tissues_V8_2023_Cervix - Ectocervix Female 40-49 Up, GTEx_Tissues_V8_2023_Cervix - Endocervix Female 40-49 Up, Human_Gene_Atlas_Adipocyte, Human_Gene_Atlas_Uterus, KEGG_2021_Human_Glycosaminoglycan biosynthesis, KEGG_2021_Human_Malaria, LINCS_L1000_CRISPR_KO_Consensus_Sigs_PDE4D Down, LINCS_L1000_CRISPR_KO_Consensus_Sigs_RGS7 Down, TF_Perturbations_Followed_by_Expression_FLI1 KD HUMAN GSE27524 CREEDSID GENE 1599 UP, TF_Perturbations_Followed_by_Expression_FLI1 KD HUMAN GSE27524 CREEDSID GENE 1596 UP, TF_Perturbations_Followed_by_Expression_FLI1 KD HUMAN GSE27524 CREEDSID GENE 1597 UP, TF_Perturbations_Followed_by_Expression_SRF KO MOUSE GSE34545 CREEDSID GENE 2884 UP, TF_Perturbations_Followed_by_Expression_FLI1 KD HUMAN GSE27524 CREEDSID GENE 1607 UP, TF_Perturbations_Followed_by_Expression_ARX SIRNA BCD HUMAN GSE73433 RNASEQ UP, TF_Perturbations_Followed_by_Expression_REST SHRNA C2 HUMAN GSE90068 PBMPA RNASEQ DOWN"
LV5,"CellMarker_2024_Basal Cell Bladder Human, CellMarker_2024_Basal Cell Lung Human, CellMarker_2024_Keratinocyte Skin Human, GO_Biological_Process_2025_Intermediate Filament Organization (GO:0045109), GO_Biological_Process_2025_Keratinocyte Differentiation (GO:0030216), GO_Biological_Process_2025_Peptide Cross-Linking (GO:0018149),

In [8]:
summ_all_summaries  <- all_summaries %>%
    dplyr::mutate(pathway_prior = paste0(prior, '_', pathway)) %>%
    dplyr::select(LV, pathway_prior) %>%
    dplyr::group_by(LV) %>%
    dplyr::summarise(pathway_prior = paste(pathway_prior, collapse = ', '), .groups = "drop") %>%
    dplyr::arrange(as.numeric(gsub("[^0-9]", "", LV)))

saveRDS(summ_all_summaries, file = file.path(output_data_dir, "all_summaries.rds"))

In [23]:
options(repr.matrix.max.rows = Inf)

gtex_fullRes_full <- readRDS(file.path(output_data_dir, "gtex_all_CLAMP.rds"))

gtex_fullRes_full_summ  <- gtex_fullRes_full$summary  %>% 
dplyr::filter(FDR < 0.05 & AUC > 0.7)  %>% 
dplyr::group_by(LV) %>% 
dplyr::summarise(pathway_prior = paste(pathway, collapse = ', '), .groups = "drop") %>%
dplyr::arrange(as.numeric(gsub("[^0-9]", "", LV)))

gtex_fullRes_full_summ

LV,pathway_prior
<chr>,<chr>
LV1,"BP_C4-dicarboxylate Transport (GO:0015740), CellMarker_Neutrophil Nasal Polyp Human"
LV2,"GTEx_Tissues_Brain - Frontal Cortex (BA9) Male 60-69 Up, GTEx_Tissues_Brain - Frontal Cortex (BA9) Female 60-69 Up, GTEx_Tissues_Brain - Cortex Male 30-39 Up, GTEx_Tissues_Brain - Frontal Cortex (BA9) Male 20-29 Up, GTEx_Tissues_Brain - Frontal Cortex (BA9) Male 70-79 Up, GTEx_Tissues_Brain - Frontal Cortex (BA9) Female 50-59 Up, GTEx_Tissues_Brain - Frontal Cortex (BA9) Male 40-49 Up, GTEx_Tissues_Brain - Anterior Cingulate Cortex (BA24) Female 20-29 Up"
LV3,"GTEx_Tissues_Whole Blood Female 50-59 Up, GTEx_Tissues_Whole Blood Male 50-59 Up, GTEx_Tissues_Whole Blood Female 20-29 Up, GTEx_Tissues_Whole Blood Male 30-39 Up, GTEx_Tissues_Whole Blood Female 30-39 Up, GTEx_Tissues_Whole Blood Male 20-29 Up, GTEx_Tissues_Whole Blood Female 40-49 Up, CellMarker_Neutrophil Soft Tissue Human, CellMarker_Neutrophil Stomach Human"
LV4,"CellMarker_Endothelial Cell Aorta Mouse, CellMarker_Endothelial Cell Blood Mouse, CellMarker_Fibroblast Muscle Mouse, CellMarker_Fibroblast Skin Mouse, CellMarker_Fibroblast Undefined Mouse"
LV5,"GTEx_Tissues_Esophagus - Mucosa Male 50-59 Up, GTEx_Tissues_Esophagus - Mucosa Male 60-69 Up, GTEx_Tissues_Esophagus - Mucosa Female 60-69 Up, GTEx_Tissues_Esophagus - Mucosa Female 20-29 Up, GTEx_Tissues_Esophagus - Mucosa Male 30-39 Up, GTEx_Tissues_Esophagus - Mucosa Female 50-59 Up, GTEx_Tissues_Esophagus - Mucosa Male 20-29 Up, GTEx_Tissues_Esophagus - Mucosa Male 40-49 Up, GTEx_Tissues_Esophagus - Mucosa Female 70-79 Up"
LV6,"BP_Male Gamete Generation (GO:0048232), BP_piRNA Processing (GO:0034587), GTEx_Tissues_Testis Male 50-59 Up, GTEx_Tissues_Testis Male 60-69 Up, GTEx_Tissues_Testis Male 30-39 Up, GTEx_Tissues_Testis Male 20-29 Up, GTEx_Tissues_Testis Male 40-49 Up, GTEx_Tissues_Testis Male 70-79 Up"
LV7,"GTEx_Tissues_Muscle - Skeletal Female 60-69 Up, GTEx_Tissues_Muscle - Skeletal Male 60-69 Up, GTEx_Tissues_Muscle - Skeletal Female 70-79 Up"
LV8,"GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Male 60-69 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Female 60-69 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Female 50-59 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Male 40-49 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Male 30-39 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Male 20-29 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Female 40-49 Up, GTEx_Tissues_Skin - Sun Exposed (Lower Leg) Male 70-79 Up, CellMarker_Basal Cell Bladder Human"
LV9,"BP_Branching Involved in Blood Vessel Morphogenesis (GO:0001569), CellMarker_Arterial Cell Embryo Mouse, CellMarker_Endothelial Cell Artery Human, CellMarker_Endothelial Cell Breast Human, CellMarker_Endothelial Cell Fetal Kidney Human, CellMarker_Endothelial Cell Muscle Mouse, CellMarker_Endothelial Cell Stomach Human"
